In [ ]:
# import packages
import math
import os
import csv
from pathlib import Path
import json
import pandas as pd
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import torch
from torchinfo import summary
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
from tqdm.auto import tqdm

import sys
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root))

from utils.data_load import DataModule # used to load dataset and create dataloaders
from utils.checkpoint import ModelCheckpoint
from utils.logging import save_history

# import custom loss functions used by both VAE and diffusion
from utils.losses import (
    # for vae loss
    masked_l1_loss,
    masked_huber_loss,
    masked_l1_grad_loss,
    masked_huber_grad_loss,
    masked_multires_l1_loss,
    masked_multires_l1_grad_loss,

    # diffusion loss
    diffusion_noise_mse_loss,
    diffusion_noise_l1_loss,
    diffusion_noise_huber_loss,
    latent_l1_loss,
    latent_l2_loss,

    # evaluation metrics
    masked_mae,
    masked_rmse,
    full_mae,
    full_rmse,
    psnr,
)

/Users/ngjunhan/opt/anaconda3/envs/dl_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# set device to gpu when available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# root_dir = Path("/content/drive/MyDrive/music_inpainting_project/diffusion_model")
# root_dir.mkdir(parents=True, exist_ok=True)

### **Load Data**

In [ ]:
# load in dataset and create dataloaders using data_load.datamodule class
# using datamodule class to load data
dm = DataModule(
    repo_id="han2o/grant-ortsaem-processedV3", # hugging face repo id for dataset
    variant="long_gaps",                      # short_gap (0.5 to 2.0) or long_gap(0.5 to 5.0)
    input_key="masked_spectrogram",
    target_key="spectrogram",
    mask_key="mask",
    batch_size=16,                             # batch size for dataloaders
    num_workers=0,
    streaming=True,
    # maximum number of training/ validation/ test samples. Set to None to use the entire dataset.
    max_train_samples=10000, 
    max_val_samples=1000,
    max_test_samples=1000, 
)

train_loader, val_loader, test_loader = dm.setup()

In [ ]:
# inspect one batch of data
batch = next(iter(train_loader))

print("keys:", batch.keys())
print("x shape:", batch["x"].shape)
print("y shape:", batch["y"].shape)
print("mask shape:", batch["mask"].shape)

if "gap_seconds" in batch:
    print("example gap:", batch["gap_seconds"][0])

In [ ]:
# visualise a sample spectrogram, mask, and target
x = batch["x"][0, 0].cpu().numpy()
y = batch["y"][0, 0].cpu().numpy()
m = batch["mask"][0, 0].cpu().numpy()

plt.figure(figsize=(25, 8))

plt.subplot(1, 3, 1)
plt.imshow(x, aspect="auto", origin="lower")
plt.title("Masked Spectrogram (x)", weight="bold")

plt.subplot(1, 3, 2)
plt.imshow(y, aspect="auto", origin="lower")
plt.title("Clean Spectrogram (y)", weight="bold")

plt.subplot(1, 3, 3)
plt.imshow(m, aspect="auto", origin="lower")
plt.title("Mask", weight="bold")

plt.tight_layout()
plt.show()

---

### **Building VAE**

VAE is introduced to learn a compressed latent representation of the spectorgram. The input spectrogram may be very large in spatial size, so the U-Net becomes expensive in memory to compute. 

We first encode the spectrogram into smaller latent tensor, ensuring that the latent space preserves the important structure. We then proceed to decode the latent space back into the original spectrogram. So the VAE is learning a mapping:
$$
x \rightarrow z \rightarrow \hat{x}
$$

---
For each of these input $x$, the encoder predicts the parameters of a Gaussian latent distribution 
$$
q_\phi(z \mid x) = \mathcal{N}\bigl(z;\mu_\phi(x), \operatorname{diag}(\sigma_\phi^2(x))\bigr)
$$

Hence the encoder outputs, $\mu = \mu_\phi(x)$ and $\log \sigma^2 = \text{logvar}_{\phi(x)}$.

Then, the latent sample is obtained using the reparameterisation trick:
$$
z = \mu + \sigma * \epsilon
$$

where $\sigma = \exp\left(\frac{1}{2} \text{logvar}\right), \qquad \epsilon \sim \mathcal{N}(0, I)$

The decoder then reconstructs:
$$
\hat{x} = p_\theta(x \mid z)
$$
---

The VAE architecture has 3 main blocks:
- Encoder
- Latent Sampling
- Decoder

---

#### **VAE Helper Functions and Class Utilities**

In [ ]:
# padding helper function
# VAE and U-NEt will downsample by powers of 2, with 4 downsampling stage, both height and width shoudl be dividable by 16 to avoid size mismatch issues
# padding is added to the right and bottom of the spectrogram, and removed after model output

# pad the last two dimensions (H, W) so they are divisible by the specified multiple
def padding(x, multiple=16):

    # padding the last 2 dim
    b, c, h, w = x.shape

    # amount of padding needed for height
    pad_h = (multiple -( h % multiple)) % multiple
    # amount of padding needed for width
    pad_w = (multiple -( w % multiple)) % multiple

    # pad format for 4d tensor
    x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode="constant", value=0.0)

    pad_info = {
        "orig_h": h,
        "orig_w": w,
        "pad_h": pad_h,
        "pad_w": pad_w,
    }
    return x_pad, pad_info

# remove padding that was added
def unpadding(x, pad_info):
    return x[..., :pad_info["orig_h"], :pad_info["orig_w"]]


In [ ]:
# sinusoidal timestamp embedding
# a key element of diffusion model, is to condition on the current timestep in each diffusion step
# this function generates a sinusoidal embedding for a given timestep, which can be added to the model's input to provide temporal information

# convert integer timestep into sinusoidal embedding vector
class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    # create sinusoidal embeddings like in diffusion models
    def forward(self, t):
        half_dim = self.dim // 2

        # create frequency scale
        freq_factor = math.log(10000) / max(half_dim - 1, 1)
        freqs = torch.exp(
            torch.arange(half_dim, device=t.device, dtype=torch.float32) * (-freq_factor)
        )

        # shape: [B, half_dim]
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)

        # concatenate sin and cos
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)

        # if dim is odd, pad one extra feature
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))

        return emb

In [ ]:
# VAE architecture blocks for spectrogram denoising

# Convblock -> GroupNorm + SiLu activation
class ConvGNSiLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, groups=8):
        super().__init__()

        # convolutional layer
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )

        # GroupNorm is more stable than BatchNorm for diffusion
        self.norm = nn.GroupNorm(num_groups=min(groups, out_channels), num_channels=out_channels)

        # SiLU is a smooth nonlinearity commonly used in modern diffusion models
        self.act = nn.SiLU()

    # forward pass
    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x

# 2D residual blocks with toggle time embedding conditioning
# residual block is used to that the model learns to keep onlyu useful information and change what is needed simialr to resnet
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=None, groups=8):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.time_emb_dim = time_emb_dim

        # first normalization + convolution path
        self.norm1 = nn.GroupNorm(num_groups=min(groups, in_channels), num_channels=in_channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        # if we are using timestep conditioning, project time embedding to channel size
        if time_emb_dim is not None:
            self.time_proj = nn.Linear(time_emb_dim, out_channels)
        else:
            self.time_proj = None

        # second normalization + convolution path
        self.norm2 = nn.GroupNorm(num_groups=min(groups, out_channels), num_channels=out_channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        # if input/output channels differ, use a 1x1 conv for residual path
        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.skip = nn.Identity()

    # forward pass
    def forward(self, x, t_emb=None):
        """
        x: [B, C, H, W]
        t_emb: [B, time_emb_dim]
        """
        residual = self.skip(x)

        # first conv path
        h = self.norm1(x)
        h = self.act1(h)
        h = self.conv1(h)

        # inject time embedding after the first conv
        if self.time_proj is not None and t_emb is not None:
            # project [B, time_emb_dim] -> [B, out_channels]
            t_out = self.time_proj(t_emb)

            # reshape to [B, out_channels, 1, 1] so it can broadcast spatially
            h = h + t_out[:, :, None, None]

        # second conv path
        h = self.norm2(h)
        h = self.act2(h)
        h = self.conv2(h)

        # residual addition
        return h + residual

# downsampling blocks
# reduce spatial resolution by factor of 2 using stride-2 convolution
# if input is [B, C, H, W], output will be [B, C, H/2, W/2]
class DownSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)

# updating blocks
# increase spatial resoltuion by factor of 2
# use nearest neighbor upsampling followed by a convolution to reduce checkerboard artifacts
class UpSample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        # double height and width
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.conv(x)
        return x

# attention block over spatial positions, natural u-net architecture adds attention at the bottleneck
class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"

        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.norm = nn.GroupNorm(num_groups=min(8, channels), num_channels=channels)

        # q, k, v projections
        self.to_q = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_k = nn.Conv2d(channels, channels, kernel_size=1)
        self.to_v = nn.Conv2d(channels, channels, kernel_size=1)

        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)

    # forward pass
    def forward(self, x):
        b, c, h, w = x.shape
        residual = x

        x = self.norm(x)

        q = self.to_q(x)
        k = self.to_k(x)
        v = self.to_v(x)

        # reshape to [B, heads, head_dim, H*W]
        q = q.view(b, self.num_heads, self.head_dim, h * w)
        k = k.view(b, self.num_heads, self.head_dim, h * w)
        v = v.view(b, self.num_heads, self.head_dim, h * w)

        # transpose q to [B, heads, HW, head_dim]
        q = q.permute(0, 1, 3, 2)

        # attention scores: [B, heads, HW, HW]
        attn_scores = torch.matmul(q, k) / math.sqrt(self.head_dim)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # apply attention to values
        # v: [B, heads, head_dim, HW] -> transpose to [B, heads, HW, head_dim]
        v = v.permute(0, 1, 3, 2)

        out = torch.matmul(attn_weights, v)  # [B, heads, HW, head_dim]

        # back to [B, C, H, W]
        out = out.permute(0, 1, 3, 2).contiguous().view(b, c, h, w)
        out = self.proj_out(out)

        return out + residual


#### **Convolutional Spatial Encoder**

Encoder takes a 1 channel spectrogram and converts it into a low-resolution latent representation

The encoder first projects from 1 channel (B, 1, H, W) to larger channel dimensions (B, 64, H, W) for richer feature space learning.

It then applies 4 resolution reduction stages, where each stage contains a residual block and then a downsampling layer, which halfs the height and width of the input feature dimensions. While the feature dimensions reduces the encoder channel width grows as depth increase. $First: 64 \rightarrow Second: 128 \rightarrow Third: 256 \rightarrow Forth: 256$

At the bottom of the encoder, we use a bottleneck attention that consists of:
$$
Residual \rightarrow Self-Attention \rightarrow Residual
$$

The model can then attend across different spatial locations of the compressed spectrogram, allowing the encoder to not only use local convolutional features, but also model longer range structures. This can help capture repeated harmonic patterns, broader time context and relationships between different frequency regions.

Lastly, we finish the build with 2 $1 \text{x} 1$ convolution heads. Hence the encoder produces:
$$
\mu \in \mathbb{R}^{B \times 4 \times H' \times W'}
$$

and 

$$ 
logvar \in \mathbb{R}^{B \times 4 \times H' \times W'}

In [ ]:
# building VAE
# encoder class, encodes spectrogram into a latent representation with mean and variance for reparameterisation
class Encoder(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, latent_channels=4):
        super().__init__()

        # initial projection from 1 channel to base feature channels
        self.in_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        # downsampling stack
        self.block1 = nn.Sequential(
            ResidualBlock(base_channels, base_channels),
            ResidualBlock(base_channels, base_channels),
        )
        self.down1 = DownSample(base_channels)

        self.block2 = nn.Sequential(
            ResidualBlock(base_channels, base_channels * 2),
            ResidualBlock(base_channels * 2, base_channels * 2),
        )
        self.down2 = DownSample(base_channels * 2)

        self.block3 = nn.Sequential(
            ResidualBlock(base_channels * 2, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )
        self.down3 = DownSample(base_channels * 4)

        self.block4 = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )
        self.down4 = DownSample(base_channels * 4)

        # bottleneck
        self.mid = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            SelfAttention(base_channels * 4, num_heads=4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )

        # mean and log-variance heads
        self.mu_head = nn.Conv2d(base_channels * 4, latent_channels, kernel_size=1)
        self.logvar_head = nn.Conv2d(base_channels * 4, latent_channels, kernel_size=1)

    # forward pass through encoder
    def forward(self, x):
        x = self.in_conv(x)

        x = self.block1(x)
        x = self.down1(x)

        x = self.block2(x)
        x = self.down2(x)

        x = self.block3(x)
        x = self.down3(x)

        x = self.block4(x)
        x = self.down4(x)

        x = self.mid(x)

        mu = self.mu_head(x)
        logvar = self.logvar_head(x)

        return mu, logvar

#### **Convolutional Spatial Decoder**

The decoder mirrors the encoder. It takes the latent tensor $z$ and reconstructs the spectrogram.

From the compact latent tensor, it projects the space into richer feature dimensions. Just like the encoder, the decoder contains the same bottleneck. So the reconstruction uses the global interactions as well.

Upsampling uses the nearest neighbour interpolation followed by convolution. This is chosen because it reduces the checkerboard artifects. 


In [ ]:
# decoder class, decodes the latent representation back to a spectrogram
class Decoder(nn.Module):
    def __init__(self, out_channels=1, base_channels=64, latent_channels=4):
        super().__init__()

        # first project latent channels into feature channels
        self.in_conv = nn.Conv2d(latent_channels, base_channels * 4, kernel_size=3, padding=1)

        # bottleneck
        self.mid = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            SelfAttention(base_channels * 4, num_heads=4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )

        # upsampling stack
        self.block4 = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 4),
        )
        self.up4 = UpSample(base_channels * 4)

        self.block3 = nn.Sequential(
            ResidualBlock(base_channels * 4, base_channels * 4),
            ResidualBlock(base_channels * 4, base_channels * 2),
        )
        self.up3 = UpSample(base_channels * 2)

        self.block2 = nn.Sequential(
            ResidualBlock(base_channels * 2, base_channels * 2),
            ResidualBlock(base_channels * 2, base_channels),
        )
        self.up2 = UpSample(base_channels)

        self.block1 = nn.Sequential(
            ResidualBlock(base_channels, base_channels),
            ResidualBlock(base_channels, base_channels),
        )
        self.up1 = UpSample(base_channels)

        # final output conv back to 1 spectrogram channel
        self.out_norm = nn.GroupNorm(num_groups=min(8, base_channels), num_channels=base_channels)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=3, padding=1)

    # forward pass
    def forward(self, z):
        x = self.in_conv(z)
        x = self.mid(x)

        x = self.block4(x)
        x = self.up4(x)

        x = self.block3(x)
        x = self.up3(x)

        x = self.block2(x)
        x = self.up2(x)

        x = self.block1(x)
        x = self.up1(x)

        x = self.out_norm(x)
        x = self.out_act(x)
        x = self.out_conv(
            x)


        return x

In [ ]:
# full VAE model combining encoder and decoder
class VAE(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, latent_channels=4):
        super().__init__()

        self.encoder = Encoder(
            in_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels,
        )

        self.decoder = Decoder(
            out_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels,
        )

        self.latent_channels = latent_channels

    # reutrn μ and log(σ^2) for latent distribution
    def encode(self, x):
        mu, logvar = self.encoder(x)
        return mu, logvar

    # sample z using reparameterisation trick
    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    # decode latent z into a reconstructed spectrogram
    def decode(self, z):
        return self.decoder(z)

    # full VAE forwas pass
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z


In [ ]:
# select vae loss
def get_loss(variant="long_gap",
             shortgap_loss="masked_l1_grad",
             longgap_loss="masked_multires_l1_grad"
             ):
    if variant == "short_gap":
        loss = shortgap_loss
    elif variant == "long_gap":
        loss = longgap_loss
    else:
        raise ValueError(
            f"Invalid gap variant {variant}."
        )

    return loss

# compute only the reconstruction part of the VAE loss
def compute_reconloss(recon, target, mask, lossfn="masked_l1_grad",
                      context_weight=0.1, grad_weight=0.1, delta=1.0, scales=(1, 2, 4),
                      scale_weights=None, eps=1e-8,
                      ):

    if lossfn == "masked_l1":
        total_loss, gap_loss, context_loss = masked_l1_loss(
            pred=recon,
            target=target,
            mask=mask,
            context_weight=context_weight,
            eps=eps,
        )
        grad_loss = torch.tensor(0.0, device=recon.device)

    elif lossfn == "masked_huber":
        total_loss, gap_loss, context_loss = masked_huber_loss(
            pred=recon,
            target=target,
            mask=mask,
            context_weight=context_weight,
            delta=delta,
            eps=eps,
        )
        grad_loss = torch.tensor(0.0, device=recon.device)

    elif lossfn == "masked_l1_grad":
        total_loss, gap_loss, context_loss, grad_loss = masked_l1_grad_loss(
            pred=recon,
            target=target,
            mask=mask,
            context_weight=context_weight,
            grad_weight=grad_weight,
            eps=eps,
        )

    elif lossfn == "masked_huber_grad":
        total_loss, gap_loss, context_loss, grad_loss = masked_huber_grad_loss(
            pred=recon,
            target=target,
            mask=mask,
            context_weight=context_weight,
            grad_weight=grad_weight,
            delta=delta,
            eps=eps,
        )

    elif lossfn == "masked_multires_l1":
        total_loss, gap_loss, context_loss = masked_multires_l1_loss(
            pred=recon,
            target=target,
            mask=mask,
            context_weight=context_weight,
            scales=scales,
            scale_weights=scale_weights,
            eps=eps,
        )
        grad_loss = torch.tensor(0.0, device=recon.device)

    elif lossfn == "masked_multires_l1_grad":
        total_loss, gap_loss, context_loss, grad_loss = masked_multires_l1_grad_loss(
            pred=recon,
            target=target,
            mask=mask,
            context_weight=context_weight,
            grad_weight=grad_weight,
            scales=scales,
            scale_weights=scale_weights,
            eps=eps,
        )

    else:
        raise ValueError(f"Unsupported VAE reconstruction loss: {lossfn}")

    return {
        "recon_loss": total_loss,
        "gap_loss": gap_loss.detach(),
        "context_loss": context_loss.detach(),
        "grad_loss": grad_loss.detach(),
    }

In [ ]:
# full VAE loss = reconstruction + KL
def vae_loss(recon, target, mu, logvar, mask, beta_kl=1e-4, recon_lossfn="masked_l1_grad",
             context_weight=0.1, grad_weight=0.1, delta=1.0, scales=(1, 2, 4), scale_weights=None,
             eps=1e-8
             ):

    recon_dict = compute_reconloss(
        recon=recon,
        target=target,
        mask=mask,
        lossfn=recon_lossfn,
        context_weight=context_weight,
        grad_weight=grad_weight,
        delta=delta,
        scales=scales,
        scale_weights=scale_weights,
        eps=eps,
    )

    # KL divergence averaged over all latent elements
    kl_per_element = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = kl_per_element.mean()

    total_loss = recon_dict["recon_loss"] + beta_kl * kl_loss

    return {
        "loss": total_loss,
        "recon_loss": recon_dict["recon_loss"].detach(),
        "gap_loss": recon_dict["gap_loss"],
        "context_loss": recon_dict["context_loss"],
        "grad_loss": recon_dict["grad_loss"],
        "kl_loss": kl_loss.detach(),
    }

In [ ]:
# training utilities
def get_lr(optimiser):
  """
  get current learning rate from optimier
  """
  return optimiser.param_groups[0]["lr"]

# linear KL warmup, start with small beta increase to target beta across 10 warmup epochs
def linear_kl_warmup(epoch, target_beta=1e-4, warmup_epochs=10, start_beta=0.0):
    if warmup_epochs <= 0:
        return target_beta

    progress = min(epoch / warmup_epochs, 1.0)
    beta = start_beta + progress * (target_beta - start_beta)
    return beta


In [ ]:
# train fucntion for VAE
def trainVAE(model, dataloader, optimiser, device, beta_kl=1e-4, recon_lossfn="masked_l1_grad",
             context_weight=0.1, grad_weight=0.1, delta=1.0, scales=(1, 2, 4), scale_weights=None,
             grad_clip=1.0, use_amp=True
             ):
    model.train()

    # running averages for monitoring
    running = {
        "loss": 0.0,
        "recon_loss": 0.0,
        "gap_loss": 0.0,
        "context_loss": 0.0,
        "grad_loss": 0.0,
        "kl_loss": 0.0,
        "gap_mae": 0.0,
        "gap_rmse": 0.0,
        "full_mae": 0.0,
        "full_rmse": 0.0,
        "psnr": 0.0,
    }
    n_batches = 0

    use_amp = use_amp and ("cuda" in str(device))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for batch in tqdm(dataloader, desc="Training VAE", leave=False):
       # masked input spectrogram
        # x = batch["x"].to(device, non_blocking=True)

        # clean target spectrogram
        y = batch["y"].to(device, non_blocking=True)

        # binary inpainting mask: 1 = missing region, 0 = known region
        m = batch["mask"].to(device, non_blocking=True)

        # pad input, target, and mask to the same divisible shape
        # x, _ = padding(x, multiple=16)
        y, pad_info= padding(y, multiple=16)
        m, _ = padding(m, multiple=16)

        optimiser.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
          # forward pass:
          # vae learns a clean latent representation
          recon, mu, logvar, z = model(y)

          # compute the full VAE loss
          loss_dict = vae_loss(
              recon=recon,
              target=y,
              mu=mu,
              logvar=logvar,
              mask=m,
              beta_kl=beta_kl,
              recon_lossfn=recon_lossfn,
              context_weight=context_weight,
              grad_weight=grad_weight,
              delta=delta,
              scales=scales,
              scale_weights=scale_weights,
          )

        # backpropagate
        scaler.scale(loss_dict["loss"]).backward()

        # optional gradient clipping for stability
        # optional gradient clipping
        if grad_clip is not None:
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        scaler.step(optimiser)
        scaler.update()

        recon_eval = unpadding(recon.detach(), pad_info)
        y_eval = unpadding(y, pad_info)
        m_eval = unpadding(m, pad_info)

        # update running values
        running["loss"] += loss_dict["loss"].item()
        running["recon_loss"] += loss_dict["recon_loss"].item()
        running["gap_loss"] += loss_dict["gap_loss"].item()
        running["context_loss"] += loss_dict["context_loss"].item()
        running["grad_loss"] += loss_dict["grad_loss"].item()
        running["kl_loss"] += loss_dict["kl_loss"].item()

        # reconstruction quality metrics
        running["gap_mae"] += masked_mae(recon_eval, y_eval, m_eval).item()
        running["gap_rmse"] += masked_rmse(recon_eval, y_eval, m_eval).item()
        running["full_mae"] += full_mae(recon_eval, y_eval).item()
        running["full_rmse"] += full_rmse(recon_eval, y_eval).item()
        running["psnr"] += psnr(recon_eval, y_eval).item()

        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}


# evaluation function for VAE, similar to training loop but without backpropagation and with model in eval mode
@torch.no_grad()
def evalVAE(model, dataloader, device, beta_kl=1e-4, recon_lossfn="masked_l1_grad",
            context_weight=0.1, grad_weight=0.1, delta=1.0, scales=(1, 2, 4), scale_weights=None,
            use_amp=True
            ):
    model.eval()

    running = {
        "loss": 0.0,
        "recon_loss": 0.0,
        "gap_loss": 0.0,
        "context_loss": 0.0,
        "grad_loss": 0.0,
        "kl_loss": 0.0,
        "gap_mae": 0.0,
        "gap_rmse": 0.0,
        "full_mae": 0.0,
        "full_rmse": 0.0,
        "psnr": 0.0,
    }
    n_batches = 0

    use_amp = use_amp and ("cuda" in str(device))

    for batch in tqdm(dataloader, desc="Eval VAE", leave=False):
        # masked input spectrogram
        # x = batch["x"].to(device, non_blocking=True)

        # clean target spectrogram
        y = batch["y"].to(device, non_blocking=True)

        # binary inpainting mask
        m = batch["mask"].to(device, non_blocking=True)

        # pad all tensors the same way
        # x, _ = padding(x, multiple=16)
        y, pad_info = padding(y, multiple=16)
        m, _ = padding(m, multiple=16)

        with torch.amp.autocast("cuda", enabled=use_amp):
          # forward pass on masked input
          recon, mu, logvar, z = model(y)

          loss_dict = vae_loss(
                recon=recon,
                target=y,
                mu=mu,
                logvar=logvar,
                mask=m,
                beta_kl=beta_kl,
                recon_lossfn=recon_lossfn,
                context_weight=context_weight,
                grad_weight=grad_weight,
                delta=delta,
                scales=scales,
                scale_weights=scale_weights,
            )

        recon_eval = unpadding(recon.detach(), pad_info)
        y_eval = unpadding(y, pad_info)
        m_eval = unpadding(m, pad_info)

        running["loss"] += loss_dict["loss"].item()
        running["recon_loss"] += loss_dict["recon_loss"].item()
        running["gap_loss"] += loss_dict["gap_loss"].item()
        running["context_loss"] += loss_dict["context_loss"].item()
        running["grad_loss"] += loss_dict["grad_loss"].item()
        running["kl_loss"] += loss_dict["kl_loss"].item()

        running["gap_mae"] += masked_mae(recon_eval, y_eval, m_eval).item()
        running["gap_rmse"] += masked_rmse(recon_eval, y_eval, m_eval).item()
        running["full_mae"] += full_mae(recon_eval, y_eval).item()
        running["full_rmse"] += full_rmse(recon_eval, y_eval).item()
        running["psnr"] += psnr(recon_eval, y_eval).item()

        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}

In [ ]:
# full training loop for VAE
def fit_VAE(model, train_loader, val_loader, optimiser, device,
            n_epochs, variant, checkpoint_dir, history_dir, short_recon_lossfn="masked_l1_grad",
            long_recon_lossfn="masked_multires_l1_grad", beta_target=1e-4, beta_start=0.0, use_kl_warmup=True,
            kl_warmup_epochs=10, context_weight=0.1, grad_weight=0.1, delta=1.0, scales=(1, 2, 4), scale_weights=None,
            monitor="val_gap_rmse", mode="min", patience=10, min_delta=1e-4, save_best_after_epoch=5, grad_clip=1.0,
            use_scheduler=True, scheduler_type="plateau", scheduler_factor=0.5, scheduler_patience=3, scheduler_min_lr=1e-7,

            # resume control
            start_epoch=1,
            resume_checkpoint_path=None,
            load_history=True,
            resume_scheduler=False,

            ):
    checkpoint_dir = Path(checkpoint_dir)
    history_dir = Path(history_dir)

    # choose the correct reconstruction loss for this gap size
    recon_lossfn = get_loss(
        variant=variant,
        shortgap_loss=short_recon_lossfn,
        longgap_loss=long_recon_lossfn,
    )

    print(f"Using VAE reconstruction loss: {recon_lossfn} for gap {variant}")

    # history dictionary
    history = {
        "epoch": [],
        "beta_kl": [],
        "lr": [],

        "train_loss": [],
        "train_recon_loss": [],
        "train_kl_loss": [],
        "train_gap_loss": [],
        "train_context_loss": [],
        "train_grad_loss": [],
        "train_gap_mae": [],
        "train_gap_rmse": [],
        "train_full_mae": [],
        "train_full_rmse": [],
        "train_psnr": [],

        "val_loss": [],
        "val_recon_loss": [],
        "val_kl_loss": [],
        "val_gap_loss": [],
        "val_context_loss": [],
        "val_grad_loss": [],
        "val_gap_mae": [],
        "val_gap_rmse": [],
        "val_full_mae": [],
        "val_full_rmse": [],
        "val_psnr": [],
    }


    # checkpoint / early stopping manager
    manager = ModelCheckpoint(
        checkpoint_dir=checkpoint_dir,
        monitor=monitor,
        mode=mode,
        patience=patience,
        min_delta=min_delta,
        save_best_after_epoch=save_best_after_epoch,
        verbose=True,
    )

    # resume training
    if resume_checkpoint_path is not None:
        resume_checkpoint_path = Path(resume_checkpoint_path)
        if not resume_checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {resume_checkpoint_path}")

        ckpt = torch.load(resume_checkpoint_path, map_location=device)

        # load model weights
        if "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"])
        else:
            raise KeyError("Checkpoint does not contain 'model_state_dict'")

        # load optimiser state
        if "optimiser_state_dict" in ckpt:
            optimiser.load_state_dict(ckpt["optimiser_state_dict"])

        # recover training position
        ckpt_epoch = ckpt.get("epoch", 0)
        start_epoch = ckpt_epoch + 1

        # recover best tracking if available
        if "best_score" in ckpt:
            manager.best_score = ckpt["best_score"]
        if "best_epoch" in ckpt:
            manager.best_epoch = ckpt["best_epoch"]

        print(f"Resumed from checkpoint: {resume_checkpoint_path}")
        print(f"Checkpoint epoch: {ckpt_epoch}")
        print(f"Will continue from epoch: {start_epoch}")
        print(f"Recovered best score: {manager.best_score}")
        print(f"Recovered best epoch: {manager.best_epoch}")

    # continue from old history csv
    if load_history and history_dir.exists():
        old_history_df = pd.read_csv(history_dir)

        # only load columns that match current history keys
        missing_cols = [k for k in history.keys() if k not in old_history_df.columns]
        if len(missing_cols) == 0:
            history = {col: old_history_df[col].tolist() for col in old_history_df.columns}
            print(f"Loaded existing history from: {history_dir}")
            print(f"Existing history length: {len(old_history_df)} epochs")
        else:
            print("History file exists but columns do not fully match current format.")
            print("Starting a fresh history dictionary instead.")

    # build scheduler
    scheduler = None
    if use_scheduler:
        if scheduler_type == "plateau":
            scheduler = ReduceLROnPlateau(
                optimiser,
                mode=mode,                 # "min" for val_gap_rmse
                factor=scheduler_factor,   # multiply LR by this when plateaued
                patience=scheduler_patience,
                min_lr=scheduler_min_lr,
            )
        elif scheduler_type == "cosine":
          remaining_epochs = max(n_epochs - start_epoch + 1, 1)
          scheduler_tmax = remaining_epochs if resume_scheduler else n_epochs

          scheduler = CosineAnnealingLR(
              optimiser,
              T_max=scheduler_tmax,
              eta_min=scheduler_min_lr,
          )
        else:
            raise ValueError(f"Unsupported scheduler_type: {scheduler_type}")

    # epoch loop
    for epoch in tqdm(range(start_epoch, n_epochs + 1), desc="Training VAE"):
        # choose KL beta for this epoch
        if use_kl_warmup:
            beta_this_epoch = linear_kl_warmup(
                epoch=epoch,
                target_beta=beta_target,
                warmup_epochs=kl_warmup_epochs,
                start_beta=beta_start,
            )
        else:
            beta_this_epoch = beta_target

        # train one epoch
        train_metrics = trainVAE(
            model=model,
            dataloader=train_loader,
            optimiser=optimiser,
            device=device,
            beta_kl=beta_this_epoch,
            recon_lossfn=recon_lossfn,
            context_weight=context_weight,
            grad_weight=grad_weight,
            delta=delta,
            scales=scales,
            scale_weights=scale_weights,
            grad_clip=grad_clip,
        )

        # validate one epoch
        val_metrics = evalVAE(
            model=model,
            dataloader=val_loader,
            device=device,
            beta_kl=beta_this_epoch,
            recon_lossfn=recon_lossfn,
            context_weight=context_weight,
            grad_weight=grad_weight,
            delta=delta,
            scales=scales,
            scale_weights=scale_weights,
        )

        # scheduler step
        if scheduler is not None:
            if scheduler_type == "plateau":
                if monitor == "val_gap_rmse":
                    scheduler.step(val_metrics["gap_rmse"])
                elif monitor == "val_loss":
                    scheduler.step(val_metrics["loss"])
                else:
                    # fallback: use val loss if monitor is not directly mapped
                    scheduler.step(val_metrics["loss"])
            elif scheduler_type == "cosine":
                scheduler.step()

        current_lr = get_lr(optimiser)

        # save everything for this epoch
        epoch_record = {
            "epoch": epoch,
            "beta_kl": beta_this_epoch,
            "lr": current_lr,

            "train_loss": train_metrics["loss"],
            "train_recon_loss": train_metrics["recon_loss"],
            "train_kl_loss": train_metrics["kl_loss"],
            "train_gap_loss": train_metrics["gap_loss"],
            "train_context_loss": train_metrics["context_loss"],
            "train_grad_loss": train_metrics["grad_loss"],
            "train_gap_mae": train_metrics["gap_mae"],
            "train_gap_rmse": train_metrics["gap_rmse"],
            "train_full_mae": train_metrics["full_mae"],
            "train_full_rmse": train_metrics["full_rmse"],
            "train_psnr": train_metrics["psnr"],

            "val_loss": val_metrics["loss"],
            "val_recon_loss": val_metrics["recon_loss"],
            "val_kl_loss": val_metrics["kl_loss"],
            "val_gap_loss": val_metrics["gap_loss"],
            "val_context_loss": val_metrics["context_loss"],
            "val_grad_loss": val_metrics["grad_loss"],
            "val_gap_mae": val_metrics["gap_mae"],
            "val_gap_rmse": val_metrics["gap_rmse"],
            "val_full_mae": val_metrics["full_mae"],
            "val_full_rmse": val_metrics["full_rmse"],
            "val_psnr": val_metrics["psnr"],
        }

        # append to history
        for key in history:
            history[key].append(epoch_record[key])

        # save history csv every epoch
        history_df = pd.DataFrame(history)
        history_df.to_csv(history_dir, index=False)

        # print summary
        print(f"Epoch {epoch:02d}")
        print(f"KL Beta:          {beta_this_epoch:.8f}")
        print(f"Learning Rate:    {current_lr:.8e}")
        print(f"Train Loss:       {train_metrics['loss']:.6f}")
        print(f"Val Loss:         {val_metrics['loss']:.6f}")
        print(f"Train Recon Loss: {train_metrics['recon_loss']:.6f}")
        print(f"Val Recon Loss:   {val_metrics['recon_loss']:.6f}")
        print(f"Train KL Loss:    {train_metrics['kl_loss']:.6f}")
        print(f"Val KL Loss:      {val_metrics['kl_loss']:.6f}")
        print(f"Train Gap RMSE:   {train_metrics['gap_rmse']:.6f}")
        print(f"Val Gap RMSE:     {val_metrics['gap_rmse']:.6f}")
        print(f"Train PSNR:       {train_metrics['psnr']:.4f}")
        print(f"Val PSNR:         {val_metrics['psnr']:.4f}")
        print("-" * 60)

        # checkpoint / early stopping step
        manager.step(
            epoch=epoch,
            metrics=epoch_record,
            model=model,
            optimiser=optimiser,
        )

        if manager.should_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    return {
        "history": history,
        "best_score": manager.best_score,
        "best_epoch": manager.best_epoch,
        "checkpoint_dir": checkpoint_dir,
        "recon_lossfn": recon_lossfn,
    }

In [ ]:
# run VAE building code
latent_channels = 4

# initialise model class
vae = VAE(
    in_channels=1,
    base_channels=32,
    latent_channels=latent_channels,
).to(device)

In [ ]:
# training loop for VAE
vae_optimiser = AdamW(vae.parameters(), lr=1e-4, weight_decay=1e-4)
vae_epochs = 150

In [ ]:
vae_checkpoint_dir = root_dir / "vae_" / "checkpoints"
vae_history_dir = root_dir / "vae_" / "history"

vae_checkpoint_dir.mkdir(parents=True, exist_ok=True)
vae_history_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
vae_results = fit_VAE(
    model=vae,
    train_loader=train_loader,
    val_loader=val_loader,
    optimiser=vae_optimiser,
    device=device,
    n_epochs=vae_epochs,
    variant="long_gap",
    checkpoint_dir=vae_checkpoint_dir,
    history_dir=vae_history_dir / "history.csv",

    # use the stronger loss for short gaps too
    short_recon_lossfn="masked_l1_grad",
    long_recon_lossfn="masked_multires_l1_grad",

    # for resume
    resume_checkpoint_path=vae_checkpoint_dir / "last_model.pt",
    load_history=True,

    # KL warmup
    beta_target=1e-4,
    beta_start=0.0,
    use_kl_warmup=False,      # set to false for resume training
    kl_warmup_epochs=10,

    # loss controls
    context_weight=0.1,
    grad_weight=0.1,
    scales=(1, 2, 4),
    scale_weights=None,

    # scheduler
    use_scheduler=True,
    scheduler_type="plateau",
    scheduler_factor=0.5,
    scheduler_patience=3,
    scheduler_min_lr=1e-7,

    # monitoring / early stop
    monitor="val_gap_rmse",
    mode="min",
    patience=5,
    min_delta=1e-4,
    save_best_after_epoch=1,

    grad_clip=1.0,
)

In [ ]:
# inspect VAE for inpainting
@torch.no_grad()
def inspectVAE(vae, dataloader, device, n_examples=4, show_mask=False):
    vae.eval()

    # take one batch from the dataloader
    batch = next(iter(dataloader))

    # masked input, clean target, and binary mask
    x = batch["x"][:n_examples].to(device)
    y = batch["y"][:n_examples].to(device)
    m = batch["mask"][:n_examples].to(device)

    # pad all tensors so they are compatible with the VAE architecture
    x_pad, pad_info = padding(x, multiple=16)
    recon_pad, mu, logvar, z = vae(x_pad)
    recon = unpadding(recon_pad, pad_info)

    final_recon = recon * m + x * (1.0 - m)

    x_np = x.cpu().numpy()
    y_np = y.cpu().numpy()
    recon_np = recon.cpu().numpy()
    final_np = final_recon.cpu().numpy()
    mask_np = m.cpu().numpy()

    n_cols = 5 if show_mask else 4
    plt.figure(figsize=(4.5 * n_cols, 4 * n_examples))

    for i in range(min(n_examples, x_np.shape[0])):
        col = 1
        # masked input
        plt.subplot(n_examples, 4, 4 * i + 1)
        plt.imshow(x_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Masked input {i}")
        plt.colorbar()

        # clean target
        plt.subplot(n_examples, 4, 4 * i + 2)
        plt.imshow(y_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Ground truth {i}")
        plt.colorbar()

        # raw model output
        plt.subplot(n_examples, 4, 4 * i + 3)
        plt.imshow(recon_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Raw VAE output {i}")
        plt.colorbar()

        # final inpainted output with known region preserved
        plt.subplot(n_examples, 4, 4 * i + 4)
        plt.imshow(final_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Final inpainted {i}")
        plt.colorbar()

        if show_mask:
            plt.subplot(n_examples, n_cols, n_cols * i + col)
            plt.imshow(mask_np[i, 0], aspect="auto", origin="lower")
            plt.title(f"Mask {i}")
            plt.colorbar()

    plt.tight_layout()
    plt.show()

In [ ]:
# best model
best_model = vae_checkpoint_dir / "best_model.pt"

In [ ]:
# load best checkpoint before inspecting
vae.load_state_dict(torch.load(best_model, map_location=device))
inspectVAE(vae, val_loader, device, n_examples=4), show_mask=True

In [ ]:
# freeze VAE once the latent space looks structurally simialar to the original
vae.load_state_dict(torch.load(best_model, map_location=device))
vae.eval()

for p in vae.parameters():
    p.requires_grad=False

---

### **Build Diffusion**

#### **Diffusion Helper Functions and Class Utilities**

In [ ]:
# helper functions to use the posterior mean instead of sampling from the posterior during diffusion training, this reduces extra randomness during training
@torch.no_grad()
# encode x to the VAE posterior mean 
def encode2latentmean(vae, x):
    mu, logvar = vae.encode(x)
    return mu 

@torch.no_grad()
# decode latent z back into spectrogram space
def decode_from_latent(vae, z):
    x_hat = vae.decode(z)
    return x_hat

We first choose a sequece of noise levels: 
$$
β_1, β_2, ..., β_T
$$

In our case, we adopted the [OpenAI's improved DDPM](https://arxiv.org/pdf/2102.09672) cosine scheduler. Instead of choosing $β_t$ directly, the it first defines:

$$
\bar{\alpha}_t = f(t)/f(0)
$$

$$f(t) = \cos^2\left(\frac{t/T + s}{1+s}\cdot \frac{\pi}{2}\right)$$

Here:
- $T$ = total number of diffusion steps
- $t \in \{0, 1, \dots, T\}$
- $s$ = small offset

Then the betas are computed from consecutive $\bar{\alpha}$ values:

$$\beta_t = 1 - \frac{\bar{\alpha}_t}{\bar{\alpha}_{t-1}}$$


Cosine scheduler, destroys the information of the spectrogram in a slower manner as compared to linear scheduler .

In [ ]:
# cosine beta scheudler class
@dataclass
class DiffusionSchedule:
    betas: torch.Tensor
    alphas: torch.Tensor
    alpha_bars: torch.Tensor
    sqrt_alpha_bars: torch.Tensor
    sqrt_one_minus_alpha_bars: torch.Tensor
    sqrt_recip_alphas: torch.Tensor
    posterior_variance: torch.Tensor

# build a cosine diffusion schedule
def cosine_schedule(num_steps, s=0.008, max_beta=0.999, device="cpu"):
    # t goes from 0 to T inclusive, so we create T+1 points
    steps = num_steps + 1
    t = torch.linspace(0, num_steps, steps, device=device, dtype=torch.float32)

    # normalized time in [0, 1]
    t = t / num_steps

    # cosine cumulative schedule f(t)
    f_t = torch.cos(((t + s) / (1 + s)) * math.pi * 0.5) ** 2

    # normalize so alpha_bar_0 = 1
    alpha_bars = f_t / f_t[0]

    # compute beta_t = 1 - alpha_bar_t / alpha_bar_{t-1}
    betas = 1.0 - (alpha_bars[1:] / alpha_bars[:-1]) # noise added at each step

    # clamp to avoid numerical issues
    betas = torch.clamp(betas, min=1e-8, max=max_beta)

    # standard DDPM quantities
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0) # cumulative product across steps

    # alpha_bars and betas allow us to generate z_t and z_0

    sqrt_alpha_bars = torch.sqrt(alpha_bars)
    sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

    # posterior variance for reverse step
    alpha_bars_prev = torch.cat(
        [torch.tensor([1.0], device=device), alpha_bars[:-1]],
        dim=0
    )
    posterior_variance = betas * (1.0 - alpha_bars_prev) / (1.0 - alpha_bars)

    return DiffusionSchedule(
        betas=betas,
        alphas=alphas,
        alpha_bars=alpha_bars,
        sqrt_alpha_bars=sqrt_alpha_bars,
        sqrt_one_minus_alpha_bars=sqrt_one_minus_alpha_bars,
        sqrt_recip_alphas=sqrt_recip_alphas,
        posterior_variance=posterior_variance,
    )

# extraction function helper
# extracts values from the 1D scheudler tensor then reshape so they will be able to broadcasts over x_shape
def extract(a, t, x_shape):
    b = t.shape[0]
    out = a.gather(0, t)

    reshape_dims = (b,) + (1,) * (len(x_shape) - 1)
    return out.view(*reshape_dims)


# forward diffusion noising q(z_t | z_0)
# sample z_t from z_0 using the closed forward process: z_t = sqrt(alpha_bar_t) * z_0 + sqrt(1-alpha_bar_t)*noise
def q_sample(z_0, t, noise, schedule):
    sqrt_alpha_bar_t = extract(schedule.sqrt_alpha_bars, t, z_0.shape)
    sqrt_one_minus_alpha_bar_t = extract(schedule.sqrt_one_minus_alpha_bars, t, z_0.shape)

    z_t = sqrt_alpha_bar_t * z_0 + sqrt_one_minus_alpha_bar_t * noise
    return z_t


# reverse sampling step p(z_{t-1}| z_t) for inferencing
def p_sample(model, z_t, z_masked_t, mask_latent, t, schedule, add_noise=True):
    # concatenate current sample, conditioning latent, and mask
    model_input = torch.cat([z_t, z_masked_t, mask_latent], dim=1)

    # predict diffusion noise
    eps_hat = model(model_input, t)

    # timestep-specific schedule values
    beta_t = extract(schedule.betas, t, z_t.shape)
    sqrt_one_minus_alpha_bar_t = extract(
        schedule.sqrt_one_minus_alpha_bars, t, z_t.shape
    )
    sqrt_recip_alpha_t = extract(schedule.sqrt_recip_alphas, t, z_t.shape)
    posterior_var_t = extract(schedule.posterior_variance, t, z_t.shape)

    # DDPM reverse mean
    model_mean = sqrt_recip_alpha_t * (
        z_t - (beta_t / sqrt_one_minus_alpha_bar_t) * eps_hat
    )

    # sample z_{t-1}
    if add_noise:
        noise = torch.randn_like(z_t)
        nonzero_mask = (t != 0).float().view(z_t.shape[0], 1, 1, 1)
        z_prev = model_mean + nonzero_mask * torch.sqrt(posterior_var_t) * noise
    else:
        z_prev = model_mean

    return z_prev


In [ ]:
# time embedding MLP; process sinusoidal time embeddings into better representations
class TimeEmbMLP(nn.Module):
    def __init__(self, time_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

    def forward(self, t_emb):
        return self.net(t_emb)

### **Build Diffusion U-Net**

Input channels are:
-  z_t (latent channel)
- z_masked (latent channel)
- mask_latent (1 channel; binary)

In [ ]:
class DiffusionU_Net(nn.Module):
    def __init__(self, latent_channels=4, base_channels=128, time_dim=256):
        super().__init__()

        # total input channels:
        # noisy latent + masked latent + 1-channel mask
        in_channels = 2 * latent_channels + 1
        out_channels = latent_channels

        self.time_embed = SinusoidalTimeEmb(time_dim)
        self.time_mlp = TimeEmbMLP(time_dim)

        # initial projection
        self.in_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        # encoder downsampling
        self.down1_block1 = ResidualBlock(base_channels, base_channels, time_emb_dim=time_dim)
        self.down1_block2 = ResidualBlock(base_channels, base_channels, time_emb_dim=time_dim)
        self.down1 = DownSample(base_channels)

        self.down2_block1 = ResidualBlock(base_channels, base_channels * 2, time_emb_dim=time_dim)
        self.down2_block2 = ResidualBlock(base_channels * 2, base_channels * 2, time_emb_dim=time_dim)
        self.down2 = DownSample(base_channels * 2)

        self.down3_block1 = ResidualBlock(base_channels * 2, base_channels * 4, time_emb_dim=time_dim)
        self.down3_block2 = ResidualBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim)
        self.down3 = DownSample(base_channels * 4)

        # bottleneck
        self.mid_block1 = ResidualBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim)
        self.mid_attn = SelfAttention(base_channels * 4, num_heads=4)
        self.mid_block2 = ResidualBlock(base_channels * 4, base_channels * 4, time_emb_dim=time_dim)

        # decoder upsampling (3 levels only)
        self.up3 = UpSample(base_channels * 4)
        self.up3_block1 = ResidualBlock(base_channels * 8, base_channels * 4, time_emb_dim=time_dim)
        self.up3_block2 = ResidualBlock(base_channels * 4, base_channels * 2, time_emb_dim=time_dim)

        self.up2 = UpSample(base_channels * 2)
        self.up2_block1 = ResidualBlock(base_channels * 4, base_channels * 2, time_emb_dim=time_dim)
        self.up2_block2 = ResidualBlock(base_channels * 2, base_channels, time_emb_dim=time_dim)

        self.up1 = UpSample(base_channels)
        self.up1_block1 = ResidualBlock(base_channels * 2, base_channels, time_emb_dim=time_dim)
        self.up1_block2 = ResidualBlock(base_channels, base_channels, time_emb_dim=time_dim)

        # output layer
        self.out_norm = nn.GroupNorm(num_groups=min(8, base_channels), num_channels=base_channels)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=3, padding=1)

    # resize x so its height and width match ref exactly
    def match_spatial(self, x, ref):
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:], mode="nearest")
        return x

    # forward pass
    def forward(self, x, t):
        # build time embedding
        t_emb = self.time_embed(t)
        t_emb = self.time_mlp(t_emb)

        # input projection
        x0 = self.in_conv(x)

        # down 1
        d1 = self.down1_block1(x0, t_emb)
        d1 = self.down1_block2(d1, t_emb)
        x1 = self.down1(d1)

        # down 2
        d2 = self.down2_block1(x1, t_emb)
        d2 = self.down2_block2(d2, t_emb)
        x2 = self.down2(d2)

        # down 3
        d3 = self.down3_block1(x2, t_emb)
        d3 = self.down3_block2(d3, t_emb)
        x3 = self.down3(d3)

        # bottleneck
        h = self.mid_block1(x3, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        # up 3
        h = self.up3(h)
        h = self.match_spatial(h, d3)
        h = torch.cat([h, d3], dim=1)
        h = self.up3_block1(h, t_emb)
        h = self.up3_block2(h, t_emb)

        # up 2
        h = self.up2(h)
        h = self.match_spatial(h, d2)
        h = torch.cat([h, d2], dim=1)
        h = self.up2_block1(h, t_emb)
        h = self.up2_block2(h, t_emb)

        # up 1
        h = self.up1(h)
        h = self.match_spatial(h, d1)
        h = torch.cat([h, d1], dim=1)
        h = self.up1_block1(h, t_emb)
        h = self.up1_block2(h, t_emb)

        # final noise prediction
        h = self.out_norm(h)
        h = self.out_act(h)
        out = self.out_conv(h)

        return out

In [ ]:
# intialise and build U-net
diffusion_unet = DiffusionU_Net(
    latent_channels=latent_channels,
    base_channels=128,
    time_dim=256,
).to(device)

print(diffusion_unet)

In [ ]:
# testing a small forward pass
# take a small batch
x_small = batch["x"][:2].to(device)
y_small = batch["y"][:2].to(device)
m_small = batch["mask"][:2].to(device)

# pad everything
x_small, _ = padding(x_small, multiple=16)
y_small, _ = padding(y_small, multiple=16)
m_small, _ = padding(m_small, multiple=16)

with torch.no_grad():
    z_clean = encode2latentmean(vae, y_small)
    z_masked = encode2latentmean(vae, x_small)

# downsample mask to latent resolution
m_latent = F.interpolate(m_small, size=z_clean.shape[-2:], mode="nearest")

# build noisy latent for a quick test
t_test = torch.randint(0, 1000, (z_clean.shape[0],), device=device)
noise_test = torch.randn_like(z_clean)

schedule = cosine_schedule(num_steps=1000, device=device)
z_t = q_sample(z_clean, t_test, noise_test, schedule)

model_input = torch.cat([z_t, z_masked, m_latent], dim=1)
eps_hat = diffusion_unet(model_input, t_test)

print("z_clean shape:", z_clean.shape)
print("z_masked shape:", z_masked.shape)
print("m_latent shape:", m_latent.shape)
print("model_input shape:", model_input.shape)
print("eps_hat shape:", eps_hat.shape)

In [ ]:
# diffusion step helper functions
# recover model's estimate of the original clean latent z_0
def predict_x0(z_t, eps_hat, t, schedule):
    sqrt_alpha_bar_t = extract(schedule.sqrt_alpha_bars, t, z_t.shape)
    sqrt_one_minus_alpha_bar_t = extract(schedule.sqrt_one_minus_alpha_bars, t, z_t.shape)

    z0_hat = (z_t - sqrt_one_minus_alpha_bar_t * eps_hat) / (sqrt_alpha_bar_t + 1e-8)
    return z0_hat

# select diffusion noise loss from losss.py
def compute_noiseloss(pred_noise, true_noise, lossfn="mse", delta=1.0):
    if lossfn == "mse":
        return diffusion_noise_mse_loss(pred_noise, true_noise)
    elif lossfn == "l1":
        return diffusion_noise_l1_loss(pred_noise, true_noise)
    elif lossfn == "huber":
        return diffusion_noise_huber_loss(pred_noise, true_noise, delta=delta)
    else:
        raise ValueError(f"Unsupported diffusion noise loss: {lossfn}")

# latent reconstruction penalty in latent space
def compute_latentloss(pred_latent, target_latent, lossfn="l1"):
    if lossfn == "l1":
        return latent_l1_loss(pred_latent, target_latent)
    elif lossfn == "l2":
        return latent_l2_loss(pred_latent, target_latent)
    else:
        raise ValueError(f"Unsupported latent auxiliary loss: {lossfn}")

In [ ]:
# given a noisey clean latent, and the masked latent and the mask, u-net predicts the noise that was added
def diffusion_step(vae, unet, schedule, batch, device, noise_loss="mse", latent_loss="l1", 
                   latent_loss_weight=0.0, delta=1.0
                   ):
    # masked spectrogram
    x = batch["x"].to(device)

    # clean spectrogram
    y = batch["y"].to(device)

    # binary mask
    m = batch["mask"].to(device)

    # pad all of them the same way
    x, _ = padding(x, multiple=16)
    y, _ = padding(y, multiple=16)
    m, _ = padding(m, multiple=16)

    # encode to latent space using frozen VAE
    with torch.no_grad():
        z_clean = encode2latentmean(vae, y)
        z_masked = encode2latentmean(vae, x)

    # resize mask to latent spatial size
    m_latent = F.interpolate(m, size=z_clean.shape[-2:], mode="nearest")

    # sample random noise
    noise = torch.randn_like(z_clean)

    # sample random diffusion step for each batch item
    t = torch.randint(
        low=0,
        high=schedule.betas.shape[0],
        size=(z_clean.shape[0],),
        device=device,
    ).long()

    # create noisy latent z_t from z_clean
    z_t = q_sample(z_clean, t, noise, schedule)

    # concatenate the inpainting information
    model_input = torch.cat([z_t, z_masked, m_latent], dim=1)

    # predict noise
    eps_hat = unet(model_input, t)

    # main diffusion objective
    noise_loss = compute_noiseloss(
        pred_noise=eps_hat,
        true_noise=noise,
        lossfn=noise_loss,
        delta=delta,
    )

    # optional auxiliary latent reconstruction objective
    if latent_loss_weight > 0.0:
        z0_hat = predict_x0(z_t, eps_hat, t, schedule)
        latent_loss = compute_latentloss(
            pred_latent=z0_hat,
            target_latent=z_clean,
            lossfn=latent_loss,
        )
    else:
        z0_hat = None
        latent_loss = torch.tensor(0.0, device=device)

    # final total loss
    total_loss = noise_loss + latent_loss_weight * latent_loss

    return {
        "loss": total_loss,
        "noise_loss": noise_loss.detach(),
        "latent_loss": latent_loss.detach(),
        "z_clean": z_clean.detach(),
        "z_masked": z_masked.detach(),
        "m_latent": m_latent.detach(),
        "z0_hat": None if z0_hat is None else z0_hat.detach(),
    }


# training fit
def trainDiffusion(vae, unet, schedule, dataloader, optimiser, device, noise_loss="mse",
                  latent_loss="l1", latent_loss_weight=0.0, delta=1.0
                  ):
    unet.train()

    running = {
        "loss": 0.0,
        "noise_loss": 0.0,
        "latent_loss": 0.0,
    }
    n_batches = 0

    for batch in tqdm(dataloader, desc="Train Diffusion", leave=False):
        optimiser.zero_grad()

        out = diffusion_step(
            vae=vae,
            unet=unet,
            schedule=schedule,
            batch=batch,
            device=device,
            noise_loss=noise_loss,
            latent_loss=latent_loss,
            latent_loss_weight=latent_loss_weight,
            delta=delta,
        )

        out["loss"].backward()

        # gradient clipping helps stabilize diffusion training
        torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=1.0)

        optimiser.step()

        running["loss"] += out["loss"].item()
        running["noise_loss"] += out["noise_loss"].item()
        running["latent_loss"] += out["latent_loss"].item()
        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}

# evaluation loop
@torch.no_grad()
def evalDiffusion(vae, unet, schedule, dataloader, device, noise_loss="mse",
                         latent_loss="l1", latent_loss_weight=0.0, delta=1.0
                         ):
    unet.eval()

    running = {
        "loss": 0.0,
        "noise_loss": 0.0,
        "latent_loss": 0.0,
    }
    n_batches = 0

    for batch in tqdm(dataloader, desc="Eval Diffusion", leave=False):
        out = diffusion_step(
            vae=vae,
            unet=unet,
            schedule=schedule,
            batch=batch,
            device=device,
            noise_loss=noise_loss,
            latent_loss=latent_loss,
            latent_loss_weight=latent_loss_weight,
            delta=delta,
        )

        running["loss"] += out["loss"].item()
        running["noise_loss"] += out["noise_loss"].item()
        running["latent_loss"] += out["latent_loss"].item()
        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}

In [ ]:
# full training loop for diffusion
def fitDiffusion(vae, unet, schedule, train_loader, val_loader, optimiser, device,
                 n_epochs, checkpoint_dir, history_dir, noise_loss="mse", latent_loss="l1",
                 latent_loss_weight=0.05, delta=1.0, monitor="val_loss", mode="min", patience=10,
                 min_delta=1e-4, save_best_after_epoch=10
                 ):
    # history dictionary storing one list per metric
    history = {
        "epoch": [],
        "train_loss": [],
        "train_noise_loss": [],
        "train_latent_loss": [],
        "val_loss": [],
        "val_noise_loss": [],
        "val_latent_loss": [],
    }

    # checkpoint manager 
    manager = ModelCheckpoint(
        checkpoint_dir=checkpoint_dir,
        monitor=monitor,
        mode=mode,
        patience=patience,
        min_delta=min_delta,
        save_best_after_epoch=save_best_after_epoch,
        verbose=True,
    )

    # main epoch loop
    for epoch in tqdm(range(1, n_epochs + 1), desc="Training Diffusion"):
        # train for one epoch
        train_metrics = trainDiffusion(
            vae=vae,
            unet=unet,
            schedule=schedule,
            dataloader=train_loader,
            optimiser=optimiser,
            device=device,
            noise_loss=noise_loss,
            latent_loss=latent_loss,
            latent_loss_weight=latent_loss_weight,
            delta=delta,
        )

        # validate for one epoch
        val_metrics = evalDiffusion(
            vae=vae,
            unet=unet,
            schedule=schedule,
            dataloader=val_loader,
            device=device,
            noise_loss=noise_loss,
            latent_loss=latent_loss,
            latent_loss_weight=latent_loss_weight,
            delta=delta,
        )

        # combine train and validation metrics into one record
        epoch_record = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_noise_loss": train_metrics["noise_loss"],
            "train_latent_loss": train_metrics["latent_loss"],
            "val_loss": val_metrics["loss"],
            "val_noise_loss": val_metrics["noise_loss"],
            "val_latent_loss": val_metrics["latent_loss"],
        }

        # append values into history
        for key in history:
            history[key].append(epoch_record[key])

        # print epoch summary
        print(f"Epoch {epoch:02d}")
        print(f"Train Loss:        {train_metrics['loss']:.6f}")
        print(f"Val Loss:          {val_metrics['loss']:.6f}")
        print(f"Train Noise Loss:  {train_metrics['noise_loss']:.6f}")
        print(f"Val Noise Loss:    {val_metrics['noise_loss']:.6f}")
        print(f"Train Latent Loss: {train_metrics['latent_loss']:.6f}")
        print(f"Val Latent Loss:   {val_metrics['latent_loss']:.6f}")
        print("-" * 60)

        # checkpoint + early stopping
        manager.step(
            epoch=epoch,
            metrics=epoch_record,
            model=unet,
            optimiser=optimiser,
        )

        # stop training if early stopping is triggered
        if manager.should_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    # save training history to CSV
    save_history(history, history_dir)

    return {
        "history": history,
        "best_score": manager.best_score,
        "best_epoch": manager.best_epoch,
        "checkpoint_dir": checkpoint_dir,
    }

In [ ]:
diffusion_optimiser = AdamW(diffusion_unet.parameters(), lr=1e-4, weight_decay=1e-4)
n_epochs = 1

In [ ]:
d_checkpoint_dir = root_dir / "diffusion_" / "checkpoints"
d_history_dir = root_dir / "diffusion_" / "history"

In [ ]:
diffusion_results = fitDiffusion(
    vae=vae,
    unet=diffusion_unet,
    schedule=schedule,
    train_loader=train_loader,
    val_loader=val_loader,
    optimiser=diffusion_optimiser,
    device=device,
    n_epochs=n_epochs,
    checkpoint_dir=d_checkpoint_dir,
    history_dir=d_history_dir,
    noise_loss_name="mse",
    latent_loss_name="l1",
    latent_loss_weight=0.05,
    delta=1.0,
    monitor="val_loss",
    mode="min",
    patience=10,
    min_delta=1e-4,
    save_best_after_epoch=10,
)

In [ ]:
# helper function to preserve known region
def preserve_region(recon_spec, x_masked, mask):
    return mask * recon_spec + (1.0 - mask) * x_masked

In [ ]:
# inferencing
# generate an inpainted latent and decode it back to a spectrogram.
@torch.no_grad()
def inferencing_latent(vae, unet, schedule, x_masked, mask, device, num_steps=None,
                       add_noise=True, return_all_steps=False):

    vae.eval()
    unet.eval()

    # default to the full diffusion length
    if num_steps is None:
        num_steps = len(schedule.betas)

    # pad inputs so spatial sizes are compatible with the model
    x_masked, pad_info = padding(x_masked, multiple=16)
    mask, _ = padding(mask, multiple=16)

    # encode masked spectrogram into latent space
    z_masked = encode2latentmean(vae, x_masked)

    # resize binary mask to latent resolution
    mask_latent = F.interpolate(mask, size=z_masked.shape[-2:], mode="nearest")

    # initialise unknown region with Gaussian noise
    z_t = torch.randn_like(z_masked)

    # optionally store latent trajectory for debugging
    latent_trajectory = []

    # reverse process from T-1 down to 0
    for step in tqdm(
        reversed(range(num_steps)),
        total=num_steps,
        desc="Sampling",
        leave=False,
    ):
        # current timestep tensor for the whole batch
        t = torch.full((z_t.shape[0],), step, device=device, dtype=torch.long)

        # build a timestep-consistent noised version of the known conditioning latent
        # q(z_t | z_0 = z_masked)
        noise_known = torch.randn_like(z_masked)
        z_masked_t = q_sample(z_masked, t, noise_known, schedule)

        # enforce known region before the model step
        # missing region comes from the current sample z_t
        # known region comes from the timestep-consistent conditioning latent z_masked_t
        z_t = mask_latent * z_t + (1.0 - mask_latent) * z_masked_t

        # one reverse diffusion step
        z_t = p_sample(
            model=unet,
            z_t=z_t,
            z_masked_t=z_masked_t,
            mask_latent=mask_latent,
            t=t,
            schedule=schedule,
            add_noise=add_noise,
        )

        # enforce known region again after the reverse step
        z_t = mask_latent * z_t + (1.0 - mask_latent) * z_masked_t

        if return_all_steps:
            latent_trajectory.append(z_t.detach().cpu())

    # decode final latent into spectrogram space
    recon = decode_from_latent(vae, z_t)

    # remove padding
    recon = unpadding(recon, pad_info)

    # preserve observed region at the spectrogram level as well
    final_recon = preserve_region(recon, x_masked[..., :recon.shape[-2], :recon.shape[-1]], mask[..., :recon.shape[-2], :recon.shape[-1]])

    result = {
        "recon": recon,
        "final_recon": final_recon,
    }

    if return_all_steps:
        result["latent_trajectory"] = latent_trajectory

    return result

In [ ]:
# inspect result
# takes one batch from thie dataloader and show the masked input, ground truth, raw reconstructd spectrogram and known-region preserved reconstrcution

@torch.no_grad()
def inspect_result(vae, unet, schedule, dataloader, device, num_steps=250, n_examples=2,
                   add_noise=True, show_mask=True):
    batch = next(iter(dataloader))

    x = batch["x"][:2].to(device)
    y = batch["y"][:2].to(device)
    m = batch["mask"][:2].to(device)

    sample_dict = inferencing_latent(
        vae=vae,
        unet=unet,
        schedule=schedule,
        x_masked=x,
        mask=m,
        device=device,
        num_steps=num_steps,
        add_noise=add_noise,
        return_all_steps=False,
    )

    recon = sample_dict["recon"]
    final_recon = sample_dict["final_recon"]

    x_np = x.cpu().numpy()
    y_np = y.cpu().numpy()
    recon_np = recon.cpu().numpy()
    final_np = final_recon.cpu().numpy()
    mask_np = m.cpu().numpy()

    n_cols = 5 if show_mask else 4

    plt.figure(figsize=(4.5 * n_cols, 4 * n_examples))

    for i in range(min(n_examples, x_np.shape[0])):
        col = 1

        # masked input
        plt.subplot(n_examples, n_cols, n_cols * i + col)
        plt.imshow(x_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Masked input {i}")
        plt.colorbar()
        col += 1

        # clean target
        plt.subplot(n_examples, n_cols, n_cols * i + col)
        plt.imshow(y_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Ground truth {i}")
        plt.colorbar()
        col += 1

        # raw decoded output
        plt.subplot(n_examples, n_cols, n_cols * i + col)
        plt.imshow(recon_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Raw recon {i}")
        plt.colorbar()
        col += 1

        # known-region-preserved output
        plt.subplot(n_examples, n_cols, n_cols * i + col)
        plt.imshow(final_np[i, 0], aspect="auto", origin="lower")
        plt.title(f"Final recon {i}")
        plt.colorbar()
        col += 1

        # mask
        if show_mask:
            plt.subplot(n_examples, n_cols, n_cols * i + col)
            plt.imshow(mask_np[i, 0], aspect="auto", origin="lower")
            plt.title(f"Mask {i}")
            plt.colorbar()

    plt.tight_layout()
    plt.show()

In [ ]:
# debuging a few intermediate reverse-diffusion steps
@torch.no_grad()
def inspect_sampling(vae, unet, schedule, dataloader, device,
                     num_steps=250, n_examples=1, snapshot_steps=(200, 100, 50, 0)
                     ):
    batch = next(iter(dataloader))

    x = batch["x"][:n_examples].to(device)
    y = batch["y"][:n_examples].to(device)
    m = batch["mask"][:n_examples].to(device)

    vae.eval()
    unet.eval()

    x_pad, pad_info = padding(x, multiple=16)
    m_pad, _ = padding(m, multiple=16)

    z_masked = encode2latentmean(vae, x_pad)
    mask_latent = F.interpolate(m_pad, size=z_masked.shape[-2:], mode="nearest")

    z_t = torch.randn_like(z_masked)
    decoded_snapshots = {}

    for step in tqdm(
        reversed(range(num_steps)),
        total=num_steps,
        desc="Sampling progress",
        leave=False,
    ):
        t = torch.full((z_t.shape[0],), step, device=device, dtype=torch.long)

        noise_known = torch.randn_like(z_masked)
        z_masked_t = q_sample(z_masked, t, noise_known, schedule)

        z_t = mask_latent * z_t + (1.0 - mask_latent) * z_masked_t

        z_t = p_sample(
            model=unet,
            z_t=z_t,
            z_masked_t=z_masked_t,
            mask_latent=mask_latent,
            t=t,
            schedule=schedule,
            add_noise=True,
        )

        z_t = mask_latent * z_t + (1.0 - mask_latent) * z_masked_t

        if step in snapshot_steps:
            recon_step = decode_from_latent(vae, z_t)
            recon_step = unpadding(recon_step, pad_info)
            decoded_snapshots[step] = recon_step.cpu().numpy()

    x_np = x.cpu().numpy()
    y_np = y.cpu().numpy()
    snapshot_keys = sorted(decoded_snapshots.keys(), reverse=True)

    plt.figure(figsize=(4 * (2 + len(snapshot_keys)), 4 * n_examples))

    for i in range(n_examples):
        # masked input
        plt.subplot(n_examples, 2 + len(snapshot_keys), i * (2 + len(snapshot_keys)) + 1)
        plt.imshow(x_np[i, 0], aspect="auto", origin="lower")
        plt.title("Masked input")
        plt.colorbar()

        # ground truth
        plt.subplot(n_examples, 2 + len(snapshot_keys), i * (2 + len(snapshot_keys)) + 2)
        plt.imshow(y_np[i, 0], aspect="auto", origin="lower")
        plt.title("Ground truth")
        plt.colorbar()

        # intermediate decoded steps
        for j, step in enumerate(snapshot_keys):
            plt.subplot(
                n_examples,
                2 + len(snapshot_keys),
                i * (2 + len(snapshot_keys)) + 3 + j
            )
            plt.imshow(decoded_snapshots[step][i, 0], aspect="auto", origin="lower")
            plt.title(f"Step {step}")
            plt.colorbar()

    plt.tight_layout()
    plt.show()

In [ ]:
inspect_result(
    vae=vae,
    unet=diffusion_unet,
    schedule=schedule,
    dataloader=val_loader,
    device=device,
    num_steps=250,
    n_examples=2,
    add_noise=True,
    show_mask=True,
)

In [ ]:
sample_dict = inferencing_latent(
    vae=vae,
    unet=diffusion_unet,
    schedule=schedule,
    x_masked=batch["x"][:2].to(device),
    mask=batch["mask"][:2].to(device),
    device=device,
    num_steps=250,
    add_noise=False,
)

In [ ]:
inspect_sampling(
    vae=vae,
    unet=diffusion_unet,
    schedule=schedule,
    dataloader=val_loader,
    device=device,
    num_steps=250,
    n_examples=1,
    snapshot_steps=(200, 100, 50, 0),
)

In [ ]:
# save checkpoints

torch.save(vae.state_dict(), "./checkpoints/vae_final.pt")
torch.save(diffusion_unet.state_dict(), "./checkpoints/unet_final.pt")

print("Saved final checkpoints.")